In [2]:
import time
from pynq.overlays.base import BaseOverlay
import socket

base = BaseOverlay("base.bit")
btns = base.btns_gpio
leds = base.leds

MY_IP_ADDRESS = "192.168.8.135"
SEAN_ADDRESS = "192.168.8.220"
LAPTOP_ADDRESS = "192.168.2.1"
PORT = 12346


# Sockets

This notebook has both a client and a server functionality. One PYNQ board in the group will be the client and SENDS the message. Another PYNQ board will be the server and RECEIVES the message.

## Server

Here, we'll build the server code to LISTEN for a message from a specific PYNQ board.

When we send/receive messages, we need to pieces of information which will tell us where to send the information. First, we need the IP address of our friend. Second, we need to chose a port to listen on. For an analogy, Alice expects her friend, Bob, to deliver a package to our back door. With this information, ALICE (server ip address) can wait at the BACK DOOR (port) for BOB (client ip address) to deliver the package.

Format of the information
 ipv4 address: ###.###.###.### (no need for leading zeros if the number is less than three digits)
 port: ##### (it could be 4 or 5 digits long, but must be >1024)
 
Use the socket documentation (Section 18.1.3) to find the appropriate functions https://python.readthedocs.io/en/latest/library/socket.html


In [3]:
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# TODO:
sock.bind((MY_IP_ADDRESS,PORT))# 1: Bind the socket to the pynq board <CLIENT-IP> at port <LISTENING-PORT>
sock.listen(5)# 2: Accept connections
print("waiting for client connection!")
(clientsocket,address)= sock.accept() # 3: Receive bytes from the connection
while True: 
        clientmsg = clientsocket.recv(2048)
        if len(clientmsg)==0:
               break
        print(clientmsg)   # 4: Print the received message
clientsocket.close()

OSError: [Errno 99] Cannot assign requested address

## Client

Now, we can implement the CLIENT code. 

Back to the analogy, now we're interested in delivering a package to our friend's back door. This means BOB (client ip address) is delivering a package to ALICE (server ip address) at her BACK DOOR (port)

**Remember to start the server before running the client code**

In [6]:
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
# TODO: 
sock.connect((LAPTOP_ADDRESS,PORT))# 1: Connect the socket (sock) to the <SERVER-IP> and choosen port <LISTENING-PORT>
for i in range(5):
    msg = f"msg num:{i}, Hello World!\n"
    sock.sendall(bytes(msg,'utf-8'))# 2: Send the message "Hello world!\n"
sock.close()# 3: Close the socket

On your server, you should see the message and then the server will shutdown! When we close a socket, both the client and the server are disconnected from the port.

Instead, change the function above to send 5 messages before closing.

The pseudocode looks like this

* connect the socket
* for i in range(5)
    * msg = input("Message to send: ")
    * send the message (msg)
* close the socket